#### Kristjan Siim

## Masinõppe abil mikropragude identifitseerimine Euler-Bernoullli talades

#### Bakalaureusetöö praktiline osa

Juhendaja:
Ljubov Jaanuska, PhD

### 1. Moodulite importimine

In [1]:
import pandas as pd
from IPython.display import Markdown
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from numpy import random
import time
from pathlib import Path

# Mudelid
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

from sklearn.model_selection import GridSearchCV

# Tehisnärvivõrgud
from keras.layers import Dense, Input, BatchNormalization
from keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow import random as tf_random

# Tulemuste analüüs
from sklearn.metrics import r2_score, root_mean_squared_error

### 2. Andmete lugemine failidest

In [2]:
folder = "./Andmed"
y_colnames = ["l1", "l2", "beta"]

# Kuna dataframe'e on vaja mitmeid erinevaid, siis paigutame nad sõnastiku-struktuuri
data = {"fr": {"name": "Sagedused"},
        "h16": {"name": "16 Haari kordajat"},
        "h32": {"name": "32 Haari kordajat"}}

# Sagedused
data["fr"]["raw"] = pd.read_csv(folder + "/" + "fr_1_3_diffused_cracking.txt", sep=r"\s+", 
                 header=None, names=y_colnames+[f"fr{i}" for i in range(1, 11)], skiprows=1)

# 16 Haari kordajat
data["h16"]["raw"] = pd.read_csv(folder + "/" + "h16_1_3_diffused_cracking.txt", sep=r"\s+", 
                 header=None, names=y_colnames+["fr1"]+[f"h{i}" for i in range(1, 17)], skiprows=1)
data["h16"]["raw"].drop("fr1", axis=1, inplace=True)   # Esimest omavõnkesagedust ei võta ennustamisel aluseks

# 32 Haari kordajat
data["h32"]["raw"] = pd.read_csv(folder + "/" + "h32_1_3_diffused_cracking.txt", sep=r"\s+", 
                 header=None, names=y_colnames+["fr1"]+[f"h{i}" for i in range(1, 33)], skiprows=1)
data["h32"]["raw"].drop("fr1", axis=1, inplace=True)   # Esimest omavõnkesagedust ei võta ennustamisel aluseks


### 3. Müra lisamine 

In [3]:
# Kopeerime eelnevalt sisse loetud ja eeltöödeldud andmestikud
keys = list(data.keys())
for i in keys:
    if "_noise" in i: continue   # Müraga andmestikule teist korda müra ei lisa
    data[i+"_noise"] = data[i].copy()
    data[i+"_noise"]["raw"] = data[i]["raw"].copy()   # Sügav koopia
    data[i+"_noise"]["name"] += " müraga"
    # Lisame sagedustele ja Haari koefitsentidele juhusliku vea normaaljaotusega 2,5%
    # Müra lisame ainult sisendmuutujatele
    cols_to_noise = [c for c in data[i]["raw"].columns if c not in y_colnames]
    noise_df = pd.DataFrame(random.normal(loc=1.0, scale=0.025, size=data[i]["raw"][cols_to_noise].shape),
                            columns=cols_to_noise, index=data[i]["raw"].index)
    data[i+"_noise"]["raw"][cols_to_noise] *= noise_df
    # Järgnev rida lisab müra kõikidele veergudele, tegelikult sihtmuutujatele pole müra vaja
    # data[i+"_noise"]["raw"] *= random.normal(loc=1.0, scale=0.025, size=data[i+"_noise"]["raw"].shape)

In [4]:
for i in data:
    display(Markdown("### *" + data[i]["name"] + "*"))
    display(data[i]["raw"].head())
    display(data[i]["raw"].describe())

### *Sagedused*

,l1,l2,beta,fr1,fr2,fr3,fr4,fr5,fr6,fr7,fr8,fr9,fr10
0,0.1,0.1,0.10,1.8628,4.6889,7.8510,10.9718,14.0870,17.2198,20.3699,23.5156,26.6439,29.7668
1,0.1,0.1,0.12,1.8601,4.6877,7.8501,10.9665,14.0761,17.2074,20.3592,23.5053,26.6304,29.7496
2,0.1,0.1,0.14,1.8574,4.6865,7.8492,10.9610,14.0649,17.1948,20.3483,23.4946,26.6163,29.7318
3,0.1,0.1,0.16,1.8547,4.6853,7.8482,10.9553,14.0535,17.1819,20.3373,23.4836,26.6016,29.7135
4,0.1,0.1,0.18,1.8520,4.6839,7.8472,10.9495,14.0417,17.1687,20.3259,23.4720,26.5862,29.6945


,l1,l2,beta,fr1,fr2,fr3,fr4,fr5,fr6,fr7,fr8,fr9,fr10
count,17220.000000,17220.000000,17220.00000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000
mean,0.360000,0.360000,0.30000,1.834459,4.514935,7.569239,10.606609,13.645348,16.684019,19.722373,22.760245,25.797550,28.834270
std,0.190793,0.190793,0.12111,0.044126,0.135397,0.214474,0.290314,0.366031,0.441423,0.516741,0.592362,0.668511,0.745201
min,0.100000,0.100000,0.10000,1.655300,4.101600,6.810100,9.467700,12.111000,14.758900,17.425200,20.113000,22.817400,25.530400
25%,0.200000,0.200000,0.20000,1.813800,4.427075,7.458600,10.462500,13.459675,16.460375,19.463050,22.462600,25.459275,28.454100
50%,0.320000,0.320000,0.30000,1.849300,4.547000,7.624100,10.685300,13.748550,16.811900,19.871500,22.930700,25.989400,29.044600
75%,0.500000,0.500000,0.40000,1.868200,4.625100,7.733900,10.826300,13.920800,17.017400,20.114000,23.209200,26.303600,29.399100
max,0.880000,0.880000,0.50000,1.876200,4.694700,7.853000,10.989800,14.125400,17.258500,20.389600,23.517200,26.643900,29.767300


### *16 Haari kordajat*

,l1,l2,beta,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16
0,0.1,0.1,0.10,0.743503,0.050822,-0.385615,0.321770,-0.277953,-0.061916,0.125699,0.183866,-0.100655,-0.150000,-0.065817,0.001337,0.047602,0.075913,0.089392,0.093290
1,0.1,0.1,0.12,0.743180,0.049882,-0.386560,0.321997,-0.277044,-0.062559,0.125694,0.184056,-0.099640,-0.150544,-0.066258,0.001126,0.047548,0.075954,0.089479,0.093390
2,0.1,0.1,0.14,0.742842,0.048904,-0.387540,0.322231,-0.276099,-0.063227,0.125688,0.184253,-0.098585,-0.151108,-0.066716,0.000907,0.047492,0.075997,0.089569,0.093494
3,0.1,0.1,0.16,0.742486,0.047886,-0.388558,0.322473,-0.275113,-0.063922,0.125682,0.184457,-0.097487,-0.151694,-0.067192,0.000678,0.047433,0.076041,0.089662,0.093601
4,0.1,0.1,0.18,0.742110,0.046825,-0.389615,0.322723,-0.274084,-0.064646,0.125674,0.184669,-0.096343,-0.152302,-0.067688,0.000440,0.047372,0.076087,0.089759,0.093712


...

,l1,l2,beta,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16
0,0.1,0.1,0.10,0.774768,0.052463,-0.383839,0.324267,-0.281230,-0.062921,0.126701,0.177194,-0.105696,-0.146508,-0.065820,0.001366,0.048182,0.075536,0.089545,0.089726
1,0.1,0.1,0.12,0.759994,0.048863,-0.399086,0.317644,-0.265188,-0.062574,0.124060,0.180073,-0.095124,-0.156191,-0.064405,0.001151,0.048404,0.077331,0.091384,0.088901
2,0.1,0.1,0.14,0.746836,0.048256,-0.404121,0.332644,-0.276445,-0.063947,0.122083,0.183204,-0.099977,-0.143762,-0.066257,0.000889,0.044436,0.077551,0.089368,0.091552
3,0.1,0.1,0.16,0.737610,0.047587,-0.398105,0.331857,-0.283988,-0.062100,0.128545,0.182238,-0.095017,-0.147377,-0.068322,0.000682,0.048054,0.079826,0.089649,0.094987
4,0.1,0.1,0.18,0.716797,0.046405,-0.385768,0.338334,-0.266770,-0.063067,0.121639,0.190788,-0.093596,-0.147850,-0.066913,0.000445,0.046801,0.075916,0.087877,0.090597


,l1,l2,beta,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16
count,17220.000000,17220.000000,17220.00000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000
mean,0.360000,0.360000,0.30000,-0.271025,0.038141,0.144211,-0.124099,0.063663,0.059768,-0.031151,-0.086255,0.018831,0.043082,0.038565,0.020507,-0.004582,-0.026229,-0.038551,-0.048967
std,0.190793,0.190793,0.12111,0.658657,0.190860,0.333977,0.312538,0.136328,0.172982,0.101038,0.250253,0.041366,0.089369,0.097101,0.077959,0.054022,0.070845,0.109433,0.137306
min,0.100000,0.100000,0.10000,-0.807046,-0.658831,-0.470412,-0.716879,-0.283988,-0.445687,-0.690363,-0.350818,-0.105696,-0.172766,-0.182918,-0.304373,-0.375560,-0.528157,-0.184395,-0.215515
25%,0.200000,0.200000,0.20000,-0.734854,-0.099463,-0.299014,-0.337994,-0.091286,-0.146498,-0.076272,-0.271500,-0.025235,-0.060745,-0.083750,-0.044136,-0.017537,-0.065030,-0.117282,-0.150364
50%,0.320000,0.320000,0.30000,-0.712435,0.102015,0.342739,-0.320057,0.120315,0.152658,-0.042348,-0.238332,0.034834,0.083571,0.093929,0.050536,0.002593,-0.054104,-0.107369,-0.128441
75%,0.500000,0.500000,0.40000,0.678525,0.194595,0.380813,0.325562,0.154672,0.185277,0.027755,0.220435,0.043592,0.106596,0.103955,0.079179,0.028539,0.049326,0.104343,0.113041
max,0.880000,0.880000,0.50000,0.801824,0.294400,0.561117,0.408922,0.344226,0.373841,0.134096,0.488652,0.138891,0.255607,0.253841,0.266987,0.241465,0.087553,0.168734,0.369134


### *32 Haari kordajat müraga*

,l1,l2,beta,h1,h2,h3,h4,h5,h6,h7,...,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32
0,0.1,0.1,0.10,0.710349,0.048687,-0.376308,0.327329,-0.261294,-0.062226,0.126706,...,-0.006275,0.007491,0.017980,0.028122,0.035043,0.040018,0.043158,0.044576,0.045000,0.044165
1,0.1,0.1,0.12,0.730293,0.048084,-0.384698,0.315332,-0.265912,-0.062206,0.125021,...,-0.006372,0.007181,0.018828,0.028718,0.036224,0.041865,0.043491,0.044302,0.046832,0.047184
2,0.1,0.1,0.14,0.734064,0.047017,-0.376130,0.318247,-0.270173,-0.062532,0.120970,...,-0.006560,0.007099,0.019443,0.028858,0.035347,0.040451,0.041438,0.047160,0.043884,0.045292
3,0.1,0.1,0.16,0.723582,0.046448,-0.395250,0.316118,-0.262193,-0.062369,0.121474,...,-0.006557,0.006638,0.018453,0.029053,0.032419,0.039389,0.044056,0.046919,0.045216,0.047260
4,0.1,0.1,0.18,0.744192,0.044990,-0.372577,0.328231,-0.267190,-0.065333,0.122674,...,-0.007021,0.007071,0.019011,0.027870,0.034087,0.040168,0.043902,0.045787,0.045973,0.046880


,l1,l2,beta,h1,h2,h3,h4,h5,h6,h7,...,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32
count,17220.000000,17220.000000,17220.00000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,...,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000,17220.000000
mean,0.360000,0.360000,0.30000,-0.267372,0.038985,0.142421,-0.122053,0.062669,0.057710,-0.030182,...,0.013807,0.006122,0.000818,-0.003797,-0.011625,-0.014337,-0.017352,-0.019958,-0.023572,-0.024500
std,0.190793,0.190793,0.12111,0.648118,0.184968,0.327691,0.306808,0.133489,0.169560,0.097508,...,0.041479,0.035461,0.028851,0.025131,0.031391,0.039486,0.049026,0.058869,0.065376,0.068881
min,0.100000,0.100000,0.10000,-0.800501,-0.623584,-0.470965,-0.653034,-0.274966,-0.433843,-0.631170,...,-0.127142,-0.168910,-0.179834,-0.179865,-0.241140,-0.259461,-0.149150,-0.082541,-0.097673,-0.107798
25%,0.200000,0.200000,0.20000,-0.723777,-0.096995,-0.292805,-0.332408,-0.088155,-0.146193,-0.074979,...,-0.027822,-0.014775,-0.006384,-0.012472,-0.026255,-0.038176,-0.051184,-0.063291,-0.071942,-0.075389
50%,0.320000,0.320000,0.30000,-0.701457,0.100684,0.337374,-0.314833,0.118396,0.150034,-0.041672,...,0.032676,0.016888,0.005851,-0.003165,-0.019300,-0.034232,-0.047671,-0.056438,-0.062001,-0.064121
75%,0.500000,0.500000,0.40000,0.666181,0.190919,0.374705,0.320080,0.152139,0.181308,0.026919,...,0.044189,0.032538,0.017762,0.012051,0.008174,0.039684,0.048866,0.053613,0.055120,0.056053
max,0.880000,0.880000,0.50000,0.782432,0.284097,0.547648,0.390152,0.335839,0.355394,0.131085,...,0.130254,0.125878,0.147375,0.147636,0.038004,0.050833,0.071062,0.102115,0.147320,0.183290


### 4. Andmehulkade eraldamine

In [5]:
# Andmeridade järjekorra segamine
for i in data:
    data[i]["shuffled"] = data[i]["raw"].sample(frac=1, random_state=39).reset_index(drop=True)

# Sisend- ja sihtmuutujate eraldamine
for i in data:
    data[i]["y"] = data[i]["shuffled"][y_colnames]
    data[i]["X"] = data[i]["shuffled"].drop(y_colnames, axis=1)

# Treening- ja testhulkade eraldamine (80% ja 20%)
for i in data:
    train_size = int(len(data[i]["shuffled"]) * 0.8)
    data[i]["X_train"] = data[i]["X"].iloc[:train_size]
    data[i]["X_test"] = data[i]["X"].iloc[train_size:]
    data[i]["y_train"] = data[i]["y"].iloc[:train_size]
    data[i]["y_test"] = data[i]["y"].iloc[train_size:]

# Sisend- ja sihtmuutujate normaliseerimine
# tehisnärvivõrkude jaoks
for i in data:
    scaler_X = MinMaxScaler().fit(data[i]["X_train"])
    data[i]["X_train_norm"] = scaler_X.transform(data[i]["X_train"])
    data[i]["X_test_norm"] = scaler_X.transform(data[i]["X_test"])
    scaler_y = MinMaxScaler().fit(data[i]["y_train"])
    data[i]["y_train_norm"] = scaler_y.transform(data[i]["y_train"])
    data[i]["y_test_norm"] = scaler_y.transform(data[i]["y_test"])
    # Säilitame skaleerijad sõnastikus, et pärast õiget kasutada
    data[i]["scaler_X"] = scaler_X
    data[i]["scaler_y"] = scaler_y

In [6]:
display(data["fr"]["y_train"].iloc[1111])
print()
display(data["fr"]["X_train"].iloc[1111])

l1      0.26
l2      0.44
beta    0.48
Name: 1111, dtype: float64

fr1      1.7692
fr2      4.2439
fr3      7.1960
fr4     10.3103
fr5     13.1944
fr6     15.9272
fr7     18.8783
fr8     21.9418
fr9     24.7989
fr10    27.6220
Name: 1111, dtype: float64

### 5. Masinõppe meetodite treenimine

In [7]:
# Kuna ka mudeleid on vaja treenida erinevaid ja mitmel andmemassiivil, siis paigutame ka mudelid sõnastiku-struktuuri
model = dict()
for i in data:
    # Iga andmemassiivi kohta sõnastik eri mudelite ennustuste jaoks
    data[i]["y_pred"] = dict()
    # Sõnastikud tulemuste jaoks
    data[i]["r2"] = dict()
    data[i]["rmse"] = dict()
    data[i]["time"] = dict()
    

#### 5.1. Lineaarne regressioon

In [8]:
model["linear"] = dict()

for i in data:
    start = time.perf_counter()
    model["linear"][i] = LinearRegression()
    
    model["linear"][i].fit(data[i]["X_train"], data[i]["y_train"])
    
    data[i]["y_pred"]["linear"] = model["linear"][i].predict(data[i]["X_test"])
    end = time.perf_counter()    

    # Tulemuste salvestamine sõnastikku
    r2 = r2_score(data[i]["y_test"], data[i]["y_pred"]["linear"], multioutput='raw_values')
    rmse = root_mean_squared_error(data[i]["y_test"], data[i]["y_pred"]["linear"], multioutput='raw_values')
    data[i]["r2"]["linear"] = r2
    data[i]["rmse"]["linear"] = rmse
    data[i]["time"]["linear"] = end-start
    
    # Tulemuste väljastamine
    print(data[i]["name"])
    print("R² per output:", r2)
    print("RMSE per output:", rmse, end="\n")
    print(f"Ajakulu: {(end-start):.2f} s\n\n")
    
    # Ennustused eraldi faili
    Path("predictions").mkdir(parents=True, exist_ok=True)   # Kui kataloogi ei eksisteeri
    with open(f"predictions/linear_{data[i]["name"]}.csv","w") as f:
        f.write("l1_test;l1_pred;l2_test;l2_pred;beta_test;beta_pred;\n")
        for j in range(len(data[i]['y_test']['l1'])):
            f.write(f"{data[i]['y_test']['l1'].iloc[j]};{data[i]['y_pred']['linear'][j][0]};")
            f.write(f"{data[i]['y_test']['l2'].iloc[j]};{data[i]['y_pred']['linear'][j][1]};")
            f.write(f"{data[i]['y_test']['beta'].iloc[j]};{data[i]['y_pred']['linear'][j][2]}\n")

    # Kirjutame logifaili
    with open("log.csv","a") as f:
        if(f.tell() == 0):  # Kui fail on tühi (just loodi), kirjutame ka päismiku
            f.write("Aeg;Meetod;Andmestik;R²(l1);R²(l2);R²(beta);RMSE(l1);RMSE(l2);RMSE(β);Ajakulu(s);Parameetrid\n")
        f.write(time.ctime() + ";Lineaarne regressioon")
        f.write(";" + data[i]["name"])
        for j in r2:
            f.write(";" + str(round(j,6)))
        for j in rmse:
            f.write(";" + str(round(j,6)))
        f.write(";" + str(round(end-start, 2)))
        f.write("\n") 

Sagedused
R² per output: [0.57822702 0.45178646 0.4466752 ]
RMSE per output: [0.12466364 0.14154434 0.08997016]
Ajakulu: 0.03 s


16 Haari kordajat
R² per output: [0.73581598 0.6004163  0.33981549]
RMSE per output: [0.09866286 0.12084301 0.09827455]
Ajakulu: 0.04 s


32 Haari kordajat
R² per output: [0.75102062 0.60955533 0.45256494]
RMSE per output: [0.0957816  0.11945309 0.08949004]
Ajakulu: 0.09 s


Sagedused müraga
R² per output: [0.24272963 0.40609278 0.39998014]
RMSE per output: [0.16704208 0.14732517 0.09368957]
Ajakulu: 0.02 s


16 Haari kordajat müraga
R² per output: [0.31336655 0.48321819 0.05808763]
RMSE per output: [0.1590607  0.13742674 0.11738529]
Ajakulu: 0.04 s


32 Haari kordajat müraga
R² per output: [0.56074555 0.54712244 0.12901897]
RMSE per output: [0.12722091 0.12864948 0.1128789 ]
Ajakulu: 0.08 s




#### 5.2. Otsustusmets

In [9]:
model["randomforest"] = dict()

for i in data:
    start = time.perf_counter()
    param_grid = {
        "n_estimators": [200, 500],
        "max_depth": [20, 40],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
        "bootstrap": [True]
    }

    model["randomforest"][i] = RandomForestRegressor(random_state=13)
    grid = GridSearchCV(
        model["randomforest"][i],
        param_grid,
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    '''
    grid.fit(data[i]["X_train"], data[i]["y_train"])
    data[i]["y_pred"]["randomforest"] = grid.predict(data[i]["X_test"])
    end = time.perf_counter()
    '''
    grid.fit(data[i]["X_train_norm"], data[i]["y_train_norm"])
    y_pred_norm = grid.predict(data[i]["X_test_norm"])
    data[i]["y_pred"]["randomforest"] = data[i]["scaler_y"].inverse_transform(y_pred_norm)
    end = time.perf_counter()        

    # Tulemuste salvestamine sõnastikku
    r2 = r2_score(data[i]["y_test"], data[i]["y_pred"]["randomforest"], multioutput='raw_values')
    rmse = root_mean_squared_error(data[i]["y_test"], data[i]["y_pred"]["randomforest"], multioutput='raw_values')
    data[i]["r2"]["randomforest"] = r2
    data[i]["rmse"]["randomforest"] = rmse
    data[i]["time"]["randomforest"] = end-start
    
    # Tulemuste väljastamine
    print(data[i]["name"])
    print(grid.best_params_)
    print("R² per output:", r2)
    print("RMSE per output:", rmse, end="\n")
    print(f"Ajakulu: {(end-start):.2f} s\n\n")

    # Ennustused eraldi faili
    Path("predictions").mkdir(parents=True, exist_ok=True)   # Kui kataloogi ei eksisteeri
    with open(f"predictions/randomforest_{data[i]["name"]}.csv","w") as f:
        f.write("l1_test;l1_pred;l2_test;l2_pred;beta_test;beta_pred;\n")
        for j in range(len(data[i]['y_test']['l1'])):
            f.write(f"{data[i]['y_test']['l1'].iloc[j]};{data[i]['y_pred']['randomforest'][j][0]};")
            f.write(f"{data[i]['y_test']['l2'].iloc[j]};{data[i]['y_pred']['randomforest'][j][1]};")
            f.write(f"{data[i]['y_test']['beta'].iloc[j]};{data[i]['y_pred']['randomforest'][j][2]}\n")

    # Kirjutame logifaili
    with open('log.csv','a') as f:
        if(f.tell() == 0):  # Kui fail on tühi (just loodi), kirjutame ka päismiku
            f.write("Aeg;Meetod;Andmestik;R²(l1);R²(l2);R²(beta);RMSE(l1);RMSE(l2);RMSE(β);Ajakulu(s);Parameetrid\n")
        f.write(time.ctime() + ";Otsustusmets")
        f.write(";" + data[i]["name"])
        for j in r2:
            f.write(";" + str(round(j,6)))
        for j in rmse:
            f.write(";" + str(round(j,6)))
        f.write(";" + str(round(end-start, 2)))
        f.write(";" + str(grid.best_params_))
        f.write("\n")

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Sagedused
{'bootstrap': True, 'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
R² per output: [0.99733301 0.99227032 0.98986874]
RMSE per output: [0.00991314 0.01680732 0.01217419]
Ajakulu: 439.90 s


Fitting 3 folds for each of 16 candidates, totalling 48 fits
16 Haari kordajat
{'bootstrap': True, 'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
R² per output: [0.99829405 0.99336165 0.98168756]
RMSE per output: [0.00792837 0.0155757  0.01636747]
Ajakulu: 774.76 s


Fitting 3 folds for each of 16 candidates, totalling 48 fits
32 Haari kordajat
{'bootstrap': True, 'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
R² per output: [0.99724422 0.99296946 0.9835864 ]
RMSE per output: [0.01007679 0.0160292  0.01549567]
Ajakulu: 1496.74 s


Fitting 3 folds for each of 16 candidates, totalling 48 fits
Sagedused müraga
{'bootstrap

#### 5.3. Gradientvõimendamine (XGBoost)

In [10]:
model["xgboost"] = dict()

for i in data:
    start = time.perf_counter()
    param_grid = {
        "n_estimators": [300, 600],
        "max_depth": [6, 10],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8],
        "colsample_bytree": [0.8],
        "gamma": [0, 1],
    }

    model["xgboost"][i] = XGBRegressor(
        objective="reg:squarederror",
        n_jobs=-1,
        tree_method="hist",
        random_state=8
    )
    grid = GridSearchCV(
        model["xgboost"][i],
        param_grid,
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    '''
    grid.fit(data[i]["X_train"], data[i]["y_train"])
    data[i]["y_pred"]["xgboost"] = grid.predict(data[i]["X_test"])
    end = time.perf_counter()
    '''
    grid.fit(data[i]["X_train_norm"], data[i]["y_train_norm"])
    y_pred_norm = grid.predict(data[i]["X_test_norm"])
    data[i]["y_pred"]["xgboost"] = data[i]["scaler_y"].inverse_transform(y_pred_norm)
    end = time.perf_counter()    


    # Tulemuste salvestamine sõnastikku
    r2 = r2_score(data[i]["y_test"], data[i]["y_pred"]["xgboost"], multioutput='raw_values')
    rmse = root_mean_squared_error(data[i]["y_test"], data[i]["y_pred"]["xgboost"], multioutput='raw_values')
    data[i]["r2"]["xgboost"] = r2
    data[i]["rmse"]["xgboost"] = rmse
    data[i]["time"]["xgboost"] = end-start
    
    # Tulemuste väljastamine
    print(data[i]["name"])
    print(grid.best_params_)
    print("R² per output:", r2)
    print("RMSE per output:", rmse, end="\n")
    print(f"Ajakulu: {(end-start):.2f} s\n")

    # Ennustused eraldi faili
    Path("predictions").mkdir(parents=True, exist_ok=True)   # Kui kataloogi ei eksisteeri
    with open(f"predictions/xgboost_{data[i]["name"]}.csv","w") as f:
        f.write("l1_test;l1_pred;l2_test;l2_pred;beta_test;beta_pred;\n")
        for j in range(len(data[i]['y_test']['l1'])):
            f.write(f"{data[i]['y_test']['l1'].iloc[j]};{data[i]['y_pred']['xgboost'][j][0]};")
            f.write(f"{data[i]['y_test']['l2'].iloc[j]};{data[i]['y_pred']['xgboost'][j][1]};")
            f.write(f"{data[i]['y_test']['beta'].iloc[j]};{data[i]['y_pred']['xgboost'][j][2]}\n")

    # Kirjutame logifaili
    with open('log.csv','a') as f:
        if(f.tell() == 0):  # Kui fail on tühi (just loodi), kirjutame ka päismiku
            f.write("Aeg;Meetod;Andmestik;R²(l1);R²(l2);R²(beta);RMSE(l1);RMSE(l2);RMSE(β);Ajakulu(s);Parameetrid\n")
        f.write(time.ctime() + ";Gradientvõimendamine (XGBoost)")
        f.write(";" + data[i]["name"])
        for j in r2:
            f.write(";" + str(round(j,6)))
        for j in rmse:
            f.write(";" + str(round(j,6)))
        f.write(";" + str(round(end-start, 2)))
        f.write(";" + str(grid.best_params_))
        f.write("\n")

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Sagedused
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 600, 'subsample': 0.8}
R² per output: [0.9975566  0.992386   0.99191374]
RMSE per output: [0.00948842 0.01668112 0.01087632]
Ajakulu: 90.75 s

Fitting 3 folds for each of 16 candidates, totalling 48 fits
16 Haari kordajat
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 600, 'subsample': 0.8}
R² per output: [0.9993866  0.99564713 0.98457944]
RMSE per output: [0.00475401 0.01261265 0.01501961]
Ajakulu: 97.83 s

Fitting 3 folds for each of 16 candidates, totalling 48 fits
32 Haari kordajat
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 600, 'subsample': 0.8}
R² per output: [0.9992217  0.99694663 0.990039  ]
RMSE per output: [0.00535519 0.01056351 0.01207148]
Ajakulu: 174.68 s

Fitting 3 folds for each of 16 candidates, totalling 48 fit

#### 5.4. Tugivektor-regressioon

In [11]:
model["svr"] = dict()

for i in data:
    start = time.perf_counter()
    param_grid = {
        "estimator__C": [1, 10, 50],
        "estimator__epsilon": [0.05, 0.1],
        "estimator__gamma": ["scale", "auto"]
    }

    model["svr"][i] = MultiOutputRegressor(SVR(kernel="rbf"))

    grid = GridSearchCV(
        model["svr"][i],
        param_grid,
        cv=3,
        n_jobs=-1,
        verbose=1
    )
    '''
    grid.fit(data[i]["X_train"], data[i]["y_train"])
    data[i]["y_pred"]["svr"] = grid.predict(data[i]["X_test"])
    end = time.perf_counter()
    '''
    grid.fit(data[i]["X_train_norm"], data[i]["y_train_norm"])
    y_pred_norm = grid.predict(data[i]["X_test_norm"])
    data[i]["y_pred"]["svr"] = data[i]["scaler_y"].inverse_transform(y_pred_norm)
    end = time.perf_counter()    

    # Tulemuste salvestamine sõnastikku
    r2 = r2_score(data[i]["y_test"], data[i]["y_pred"]["svr"], multioutput='raw_values')
    rmse = root_mean_squared_error(data[i]["y_test"], data[i]["y_pred"]["svr"], multioutput='raw_values')
    data[i]["r2"]["svr"] = r2
    data[i]["rmse"]["svr"] = rmse
    data[i]["time"]["svr"] = end-start
    
    # Tulemuste väljastamine
    print(data[i]["name"])
    print(grid.best_params_)
    print("R² per output:", r2)
    print("RMSE per output:", rmse, end="\n")
    print(f"Ajakulu: {(end-start):.2f} s\n\n")

    # Ennustused eraldi faili
    Path("predictions").mkdir(parents=True, exist_ok=True)   # Kui kataloogi ei eksisteeri
    with open(f"predictions/svr_{data[i]["name"]}.csv","w") as f:
        f.write("l1_test;l1_pred;l2_test;l2_pred;beta_test;beta_pred;\n")
        for j in range(len(data[i]['y_test']['l1'])):
            f.write(f"{data[i]['y_test']['l1'].iloc[j]};{data[i]['y_pred']['svr'][j][0]};")
            f.write(f"{data[i]['y_test']['l2'].iloc[j]};{data[i]['y_pred']['svr'][j][1]};")
            f.write(f"{data[i]['y_test']['beta'].iloc[j]};{data[i]['y_pred']['svr'][j][2]}\n")

    # Kirjutame logifaili
    with open('log.csv','a') as f:
        if(f.tell() == 0):  # Kui fail on tühi (just loodi), kirjutame ka päismiku
            f.write("Aeg;Meetod;Andmestik;R²(l1);R²(l2);R²(beta);RMSE(l1);RMSE(l2);RMSE(β);Ajakulu(s);Parameetrid\n")
        f.write(time.ctime() + ";Tugivektor-regressioon")
        f.write(";" + data[i]["name"])
        for j in r2:
            f.write(";" + str(round(j,6)))
        for j in rmse:
            f.write(";" + str(round(j,6)))
        f.write(";" + str(round(end-start, 2)))
        f.write(";" + str(grid.best_params_))
        f.write("\n")

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Sagedused
{'estimator__C': 50, 'estimator__epsilon': 0.05, 'estimator__gamma': 'scale'}
R² per output: [0.9847626  0.97412823 0.98319458]
RMSE per output: [0.02369496 0.03074898 0.01567953]
Ajakulu: 112.53 s


Fitting 3 folds for each of 12 candidates, totalling 36 fits
16 Haari kordajat
{'estimator__C': 50, 'estimator__epsilon': 0.05, 'estimator__gamma': 'scale'}
R² per output: [0.90916541 0.82964351 0.83392817]
RMSE per output: [0.05785302 0.07890363 0.04928973]
Ajakulu: 183.90 s


Fitting 3 folds for each of 12 candidates, totalling 36 fits
32 Haari kordajat
{'estimator__C': 50, 'estimator__epsilon': 0.05, 'estimator__gamma': 'scale'}
R² per output: [0.93605264 0.88247046 0.91322011]
RMSE per output: [0.04854134 0.06553769 0.03563018]
Ajakulu: 197.19 s


Fitting 3 folds for each of 12 candidates, totalling 36 fits
Sagedused müraga
{'estimator__C': 50, 'estimator__epsilon': 0.1, 'estimator__gamma': 'auto'}
R² per output: [0

#### 5.5. Tehisnärvivõrgud

In [12]:
random.seed(37)
tf_random.set_seed(14)

for i in data:
    print(data[i]["name"])
    start = time.perf_counter()
    # Kerase moodul ei toeta GridSearch'i kasutamist.
    # Parameetrite otsing on lahendatud manuaalselt.
    param_grid = {
        "units1": [128, 256],
        "units2": [64, 128],
        "lr": [0.001, 0.0005]
    }
    best_loss = float("inf")
    best_model = None

    for u1 in param_grid["units1"]:
        for u2 in param_grid["units2"]:
            for lr in param_grid["lr"]:
                print(".", end="")
                model["sequential"] = Sequential([
                    Input(shape=(data[i]["X_train_norm"].shape[1],)),
                    Dense(u1, activation="relu"),
                    Dense(u2, activation="relu"),
                    Dense(3)
                ])
                model["sequential"].compile(loss="mse", metrics=["mse"], optimizer=Adam(lr))
    
                hist = model["sequential"].fit(
                    data[i]["X_train_norm"],
                    data[i]["y_train_norm"],
                    validation_split=0.1,
                    epochs=200,
                    batch_size=32,
                    verbose=False
                )
    
                val_loss = min(hist.history["val_loss"])
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_model = model["sequential"]
                    best_params = {"units1": u1, "units2": u2, "lr": lr}

    y_pred_norm = best_model.predict(data[i]["X_test_norm"])
    data[i]["y_pred"]["sequential"] = data[i]["scaler_y"].inverse_transform(y_pred_norm)
    end = time.perf_counter()

    # Tulemuste salvestamine sõnastikku
    r2 = r2_score(data[i]["y_test"], data[i]["y_pred"]["sequential"], multioutput='raw_values')
    rmse = root_mean_squared_error(data[i]["y_test"], data[i]["y_pred"]["sequential"], multioutput='raw_values')
    data[i]["r2"]["sequential"] = r2
    data[i]["rmse"]["sequential"] = rmse
    data[i]["time"]["sequential"] = end-start
    
    # Tulemuste väljastamine
    print("\nR² per output:", r2)
    print("RMSE per output:", rmse, end="\n\n")
    print(f"Ajakulu: {(end-start):.2f} s\n")

    # Ennustused eraldi faili
    Path("predictions").mkdir(parents=True, exist_ok=True)   # Kui kataloogi ei eksisteeri
    with open(f"predictions/sequential_{data[i]["name"]}.csv","w") as f:
        f.write("l1_test;l1_pred;l2_test;l2_pred;beta_test;beta_pred;\n")
        for j in range(len(data[i]['y_test']['l1'])):
            f.write(f"{data[i]['y_test']['l1'].iloc[j]};{data[i]['y_pred']['sequential'][j][0]};")
            f.write(f"{data[i]['y_test']['l2'].iloc[j]};{data[i]['y_pred']['sequential'][j][1]};")
            f.write(f"{data[i]['y_test']['beta'].iloc[j]};{data[i]['y_pred']['sequential'][j][2]}\n")

    # Kirjutame logifaili
    with open('log.csv','a') as f:
        if(f.tell() == 0):  # Kui fail on tühi (just loodi), kirjutame ka päismiku
            f.write("Aeg;Meetod;Andmestik;R²(l1);R²(l2);R²(beta);RMSE(l1);RMSE(l2);RMSE(β);Ajakulu(s);Parameetrid\n")
        f.write(time.ctime() + ";Tehisnärvivõrgud")
        f.write(";" + data[i]["name"])
        for j in r2:
            f.write(";" + str(round(j,6)))
        for j in rmse:
            f.write(";" + str(round(j,6)))
        f.write(";" + str(round(end-start, 2)))
        f.write(";" + str(best_params))
        f.write("\n")

Sagedused
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.9947717 0.9834667 0.9879972]
RMSE per output: [0.01387968 0.02458087 0.01325104]

Ajakulu: 2099.75 s

16 Haari kordajat
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.9949251  0.98578644 0.9708242 ]
RMSE per output: [0.01367462 0.02279128 0.0206595 ]

Ajakulu: 2103.83 s

32 Haari kordajat
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.9967801  0.98674846 0.9833507 ]
RMSE per output: [0.01089234 0.0220065  0.01560655]

Ajakulu: 2101.56 s

Sagedused müraga
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.20390815 0.35946637 0.34637284]
RMSE per output: [0.17127033 0.15299897 0.09778522]

Ajakulu: 2103.05 s

16 Haari kordajat müraga
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.9879426 0.9492061 0.8869135]
RMSE per output: [0.02107793 0.04308473 0.04067371]

Ajakulu: 2101.96 s

32 Haari kordajat müraga
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

R² per output: [0.99431604